# D5b · xBD tier3: masks for the 612 exam tiles (CPU, no GPU)

Task **D5b** from `Essential_task_delegation/Progress_Tracker.md`. It prepares R12's second pool of extra training tiles (option A, chosen 19 Oct 2026), the same way D5 prepared the first. Built as a copy of `d5-xbd-converter.ipynb`, which stays unchanged.

**What it does**
1. **Tile IDs:** finds which of tier3's 6,369 post-disaster tiles DisasterM3 already uses (its segmentation data, which includes our 645 validation images), and which of the others are DisasterM3 benchmark images.
2. **Converter check:** draws the xBD polygons of the tier3 tiles DisasterM3 already uses and compares them with DisasterM3's own masks, as D5 did for xBD's first part. The lookup table should come out the same as D5's, `[0, 1, 2, 2, 3, 255]`.
3. **Converts the new benchmark tiles** (expected 612) into masks with the raw xBD codes 0–5.

**Why only the benchmark tiles (option A)**
- The new tier3 tiles that are *not* benchmark images (3,122) hold no destroyed building (1,210 buildings, all but 2 intact), so they add nothing to R12.
- The 612 benchmark tiles hold 4,549 destroyed buildings. None is one of our 645 validation images, so training on them keeps the O1 score fair.
- A model trained on them must never answer benchmark questions. R12 already follows that rule because of D5's 447 benchmark tiles; R16 with R11 answers them instead.

**Inputs on this PC** (all git-ignored except the manifests):
- tier3's labels in `dataset/xView2 Challenge Dataset - Tier 3 Data/tier3/labels/`;
- DisasterM3's masks in `dataset/DisasterM3 Dataset/DisasterM3_Instruct/masks/`;
- `train_release.json` and `support/Abrar/Implementation_DisasterM3/benchmark_release.json`.

**On Kaggle (optional):** a CPU session with the DisasterM3 mirror, Bojan Tunguz's "xView2 Challenge Dataset - Tier 3 Data" and `benchmark_release.json` attached.

**Outputs in `Champion_Task/D5b-results/`:** `masks/` (one PNG per tile), `manifest.json`, `mapping.json`, `overlap_check.csv`, `mapping_report.md`, `config.json`, `README.md`. Upload this folder for R12 together with `D5-results/`; the images come from Tunguz's tier3 copy on Kaggle.

**Done when:**
- the lookup table matches D5's;
- no converted tile is in DisasterM3's segmentation data (asserted in Cell 2);
- the counts match D5b's check of the zip: 612 tiles and 4,549 destroyed polygons.

In [1]:
# ── Cell 1: Configuration and inputs (CPU only, no installs needed) ─────────
import glob
import json
import re
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

SMOKE_TEST = False                  # True = 50 overlapping and 20 new benchmark tiles, about a minute
ON_KAGGLE = Path("/kaggle/input").exists()

def find_input(pattern):
    """First file under /kaggle/input that matches, whatever the dataset is called."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"No '{pattern}' under /kaggle/input. Attach it as described in the first cell.")
    return Path(hits[0])

if ON_KAGGLE:
    DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
    MANIFEST_PATH = DATA_ROOT / "train_release.json"
    OUT_DIR = Path("/kaggle/working/D5b")
    LABELS_DIR = Path("/kaggle/input/datasets/tunguz/xview2-challenge-dataset-tier-3-data/tier3/labels")
    BENCH_PATH = find_input("benchmark_release.json")
    D5_MAPPING = None                # the comparison with D5's table runs on the PC only
else:
    # This PC: the repo root is the folder that holds train_release.json
    REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "train_release.json").exists())
    DATA_ROOT = REPO / "dataset" / "DisasterM3 Dataset" / "DisasterM3_Instruct"
    MANIFEST_PATH = REPO / "train_release.json"
    OUT_DIR = REPO / "Essential_task_delegation" / "Champion_Task" / "D5b-results"
    LABELS_DIR = REPO / "dataset" / "xView2 Challenge Dataset - Tier 3 Data" / "tier3" / "labels"
    BENCH_PATH = REPO / "support" / "Abrar" / "Implementation_DisasterM3" / "benchmark_release.json"
    D5_MAPPING = REPO / "Essential_task_delegation" / "Champion_Task" / "D5-results" / "mapping.json"

(OUT_DIR / "masks").mkdir(parents=True, exist_ok=True)
t_start = time.time()
print(f"✓ D5b xBD tier3 on {'Kaggle' if ON_KAGGLE else 'this PC'}{'  [SMOKE TEST]' if SMOKE_TEST else ''}")
print(f"  DisasterM3:   {DATA_ROOT}")
print(f"  Tier3 labels: {LABELS_DIR}")
print(f"  Benchmark:    {BENCH_PATH}")
print(f"  Outputs:      {OUT_DIR}")


✓ D5b xBD tier3 on this PC
  DisasterM3:   D:\CSE499AB_project\dataset\DisasterM3 Dataset\DisasterM3_Instruct
  Tier3 labels: D:\CSE499AB_project\dataset\xView2 Challenge Dataset - Tier 3 Data\tier3\labels
  Benchmark:    D:\CSE499AB_project\support\Abrar\Implementation_DisasterM3\benchmark_release.json
  Outputs:      D:\CSE499AB_project\Essential_task_delegation\Champion_Task\D5b-results


## 1. Tile IDs: which tier3 tiles DisasterM3 already uses, and which are benchmark images
Same rule as D5: DisasterM3 writes `portugal_wildfire_00000123_post_disaster.png` and xBD writes `portugal-wildfire_00000123`, and both become one tile ID. A tile is **in DisasterM3** if its before- or after-image appears in the segmentation entries of `train_release.json`, which also hold the 645 validation images. The benchmark is checked separately, because H1 scores KR2 on it.

In [2]:
# ── Cell 2: Tile IDs (D5's code, pointed at tier3) ──────────────────────────
XBD_NAME = re.compile(r"^(.+?)_(\d{8})_(?:pre|post)_disaster")

def tile_id(path):
    """'train_images\\portugal_wildfire_00000123_post_disaster.png' -> 'portugal-wildfire_00000123'.
    None for images that aren't xBD tiles."""
    m = XBD_NAME.match(Path(str(path or "").replace("\\", "/")).name.replace("_sar", ""))
    return f"{m.group(1).replace('_', '-')}_{m.group(2)}" if m else None

def tiles_in(entries):
    return {tile_id(e.get(k)) for e in entries for k in ("pre_image_path", "post_image_path")} - {None}

with open(MANIFEST_PATH, encoding="utf-8") as f:
    train_entries = json.load(f)
with open(BENCH_PATH, encoding="utf-8") as f:
    bench_entries = json.load(f)
bda = [e for e in train_entries if e.get("task") == "Referring Expression Segmentation"
       and e.get("cls_description") == "Building Damage Assessment"]
seg_tiles = tiles_in(bda)              # the U-Net's 6,443 pairs, including the 645 validation images
bench_tiles = tiles_in(bench_entries)  # the benchmark that H1 scores for KR2

xbd_tiles = sorted(p.name[:-len("_post_disaster.json")] for p in LABELS_DIR.glob("*_post_disaster.json"))
overlap = [t for t in xbd_tiles if t in seg_tiles]
new = [t for t in xbd_tiles if t not in seg_tiles]
new_in_bench = [t for t in new if t in bench_tiles]   # option A: R12's second pool
new_clean = [t for t in new if t not in bench_tiles]   # no destroyed buildings (D5b's zip check): left out
assert not set(new_in_bench) & seg_tiles               # so no converted tile can be one of the 645 validation images

bdc = [e for e in bench_entries if e.get("task") == "Building Damage Counting"]
bdc_hit = sum(tile_id(e.get("post_image_path")) in set(new_in_bench) for e in bdc)
print(f"Tier3 post-disaster labels: {len(xbd_tiles):,} (expected 6,369)")
print(f"  already in DisasterM3's segmentation data: {len(overlap):,} (expected 2,635)")
print(f"  new: {len(new):,}")
print(f"    benchmark images (converted for R12):     {len(new_in_bench):,} (expected 612; "
      f"{bdc_hit:,} of {len(bdc):,} BDC benchmark questions use them)")
print(f"    not in the benchmark (left out):          {len(new_clean):,} (expected 3,122)")
if SMOKE_TEST:
    overlap, new_in_bench, new_clean = overlap[:50], new_in_bench[:20], new_clean[:20]


Tier3 post-disaster labels: 6,369 (expected 6,369)
  already in DisasterM3's segmentation data: 2,635 (expected 2,635)
  new: 3,734
    benchmark images (converted for R12):     612 (expected 612; 1,045 of 4,982 BDC benchmark questions use them)
    not in the benchmark (left out):          3,122 (expected 3,122)


## 2. Converter check on tier3
xBD stores each building as a polygon in pixel coordinates, with one of five labels. They're drawn into a mask with codes 0 (background), 1 no-damage, 2 minor-damage, 3 major-damage, 4 destroyed and 5 un-classified; worse damage is drawn last. On the tier3 tiles DisasterM3 already uses, this mask is compared with DisasterM3's 4-class mask. Each xBD label maps to the DisasterM3 class it mostly becomes, and the resulting table should match D5's. Every tier3 polygon is a single ring (checked on all 6,369 label files), so D5's rasterizer applies unchanged.

In [3]:
# ── Cell 3: xBD polygons -> mask with one code per damage label (verbatim from D5) ──
from PIL import Image, ImageDraw     # PIL, not cv2, so a local run and a Kaggle run give identical masks

SUBTYPES = ["no-damage", "minor-damage", "major-damage", "destroyed", "un-classified"]
CODE = {s: i + 1 for i, s in enumerate(SUBTYPES)}       # 0 = background, then 1-5 in this order
XBD_CODES = ["background", *SUBTYPES]
DRAW_ORDER = ["un-classified", "no-damage", "minor-damage", "major-damage", "destroyed"]  # worst damage last

def rasterize(label):
    """xBD label JSON -> (uint8 mask with codes 0-5, polygon count per label).
    Every polygon in xBD's train labels is a single ring: 'POLYGON ((x y, x y, ...))'."""
    h, w = label["metadata"].get("height", 1024), label["metadata"].get("width", 1024)
    polys = defaultdict(list)
    for f in label["features"]["xy"]:
        sub = f["properties"].get("subtype")
        if sub in CODE:
            wkt = f["wkt"]
            ring = wkt[wkt.index("((") + 2:wkt.rindex("))")]
            pts = np.round([[float(v) for v in p.split()] for p in ring.split(",")]).astype(int)
            polys[sub].append([tuple(pt) for pt in pts])
    img = Image.new("L", (w, h), 0)
    draw = ImageDraw.Draw(img)
    for sub in DRAW_ORDER:
        for p in polys[sub]:           # one polygon at a time, so a later one overwrites an earlier one
            draw.polygon(p, fill=CODE[sub])
    return np.array(img), {s: len(polys[s]) for s in SUBTYPES}

def load_label(tile):
    with open(LABELS_DIR / f"{tile}_post_disaster.json", encoding="utf-8") as f:
        return json.load(f)

# Self-check: a destroyed square inside a no-damage square keeps both labels
_toy = {"metadata": {"width": 20, "height": 20}, "features": {"xy": [
    {"properties": {"subtype": "destroyed"}, "wkt": "POLYGON ((5 5, 9 5, 9 9, 5 9, 5 5))"},
    {"properties": {"subtype": "no-damage"}, "wkt": "POLYGON ((2 2, 12 2, 12 12, 2 12, 2 2))"}]}}
_m, _n = rasterize(_toy)
assert _m[7, 7] == CODE["destroyed"] and _m[3, 3] == CODE["no-damage"] and _m[15, 15] == 0
assert _n["destroyed"] == 1 and _n["no-damage"] == 1
print("✓ Rasterizer ready (self-check passed)")


✓ Rasterizer ready (self-check passed)


In [4]:
# ── Cell 4: Converter check against DisasterM3's own masks (D5's code) ──────
# DisasterM3's masks are found and combined exactly as in Cell 4 of the D4 notebooks (read with PIL here).
def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "masks" / "masks" / filename,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

FOLDER_TO_CLASS = {
    "train_building_intact_mask": 1,
    "train_building_damaged_mask": 2,
    "train_building_destroyed_mask": 3,
}
overlap_set = set(overlap)
dm3_mask_paths = defaultdict(dict)     # tile -> {mask folder: path in the manifest}
for e in bda:
    t = tile_id(e.get("post_image_path"))
    if t in overlap_set and e.get("image_type") == "Optical":
        folder = Path(e.get("ground_truth", "").replace("\\", "/")).parent.name
        dm3_mask_paths[t][folder] = e.get("ground_truth")

def dm3_mask(tile):
    """DisasterM3's 4-class mask (0 background, 1 intact, 2 damaged, 3 destroyed); later classes overwrite."""
    combined = None
    for folder, cls in FOLDER_TO_CLASS.items():
        path = resolve_path(dm3_mask_paths[tile].get(folder))
        if path is None:
            continue
        m = np.array(Image.open(path).convert("L"))
        if combined is None:
            combined = np.zeros_like(m, dtype=np.uint8)
        combined[m > 0] = cls
    return combined

DM3_CLASSES = ["Background", "Intact", "Damaged", "Destroyed"]
table = np.zeros((len(XBD_CODES), len(DM3_CLASSES)), np.int64)   # rows: xBD code, columns: DisasterM3 class
check_rows, no_dm3_mask = [], 0
for t in tqdm(overlap, desc="Overlapping tiles"):
    dm3 = dm3_mask(t)
    if dm3 is None:
        no_dm3_mask += 1
        continue
    xbd, _ = rasterize(load_label(t))
    if xbd.shape != dm3.shape:
        xbd = np.array(Image.fromarray(xbd).resize((dm3.shape[1], dm3.shape[0]), Image.NEAREST))
    cm = np.bincount(xbd.ravel().astype(np.int64) * 4 + dm3.ravel(), minlength=24).reshape(6, 4)
    table += cm
    union = cm[1:].sum() + cm[0, 1:].sum()        # building pixels in either mask
    check_rows.append({"tile": t, "event": t.rsplit("_", 1)[0],
                       "building_iou": round(cm[1:, 1:].sum() / union, 4) if union else 1.0})
check = pd.DataFrame(check_rows)
assert len(check), "No DisasterM3 masks found for the overlapping tiles: is the mirror attached?"

# Each xBD label maps to the DisasterM3 building class it mostly becomes
lut, mapping_rows = [0], []
for c in range(1, len(XBD_CODES)):
    row = table[c]
    building_share = row[1:].sum() / max(row.sum(), 1)   # how often DisasterM3 draws it as a building at all
    cls = int(row[1:].argmax()) + 1
    class_share = row[cls] / max(row[1:].sum(), 1)       # how consistently it gets that one class
    ignore = XBD_CODES[c] == "un-classified" and (building_share < 0.5 or class_share < 0.8)
    lut.append(255 if ignore else cls)
    mapping_rows.append({"xbd_label": XBD_CODES[c], "pixels": int(row.sum()),
                         **{f"to_{n.lower()}": round(row[i] / max(row.sum(), 1), 4)
                            for i, n in enumerate(DM3_CLASSES)},
                         "maps_to": "255 (ignore)" if ignore else DM3_CLASSES[cls],
                         "consistency": round(class_share, 4)})
mapping = pd.DataFrame(mapping_rows)
agreement = sum(table[c, lut[c]] for c in range(1, 5)) / max(table[1:5].sum(), 1)   # the four graded labels

print(f"Compared {len(check):,} tiles ({no_dm3_mask} had no DisasterM3 mask)")
print(mapping.to_string(index=False))
print(f"\nPixel agreement after mapping (graded buildings): {agreement:.1%}")
print(f"Median building IoU per tile, xBD drawing vs DisasterM3: {check['building_iou'].median():.3f} "
      f"({(check['building_iou'] < 0.5).sum()} tiles below 0.5)")
for r in mapping_rows:
    if r["xbd_label"] != "un-classified" and r["consistency"] < 0.8:
        print(f"⚠ DisasterM3 is not consistent for {r['xbd_label']}: check the table before training")
print(f"Lookup table for R12 (xBD code 0-5 -> class): {lut}")

# D5b: R12 maps both pools with their own mapping.json, so the two tables should agree
same_as_d5 = None
if D5_MAPPING is not None and Path(D5_MAPPING).exists():
    d5_lut = json.load(open(D5_MAPPING))["lut"]
    same_as_d5 = d5_lut == lut
    print(f"D5's table (xBD part 1): {d5_lut} -> {'the same' if same_as_d5 else 'DIFFERENT: check before training'}")


Overlapping tiles:   0%|          | 0/2635 [00:00<?, ?it/s]

Compared 2,635 tiles (0 had no DisasterM3 mask)
    xbd_label   pixels  to_background  to_intact  to_damaged  to_destroyed      maps_to  consistency
    no-damage 63004843         0.0274     0.9726      0.0000        0.0000       Intact       0.9999
 minor-damage  5028159         0.0291     0.0008      0.9701        0.0000      Damaged       0.9991
 major-damage  3417416         0.0300     0.0003      0.9697        0.0000      Damaged       0.9997
    destroyed  3754594         0.0305     0.0001      0.0001        0.9692    Destroyed       0.9997
un-classified  1593004         0.9995     0.0004      0.0001        0.0000 255 (ignore)       0.8026

Pixel agreement after mapping (graded buildings): 97.2%
Median building IoU per tile, xBD drawing vs DisasterM3: 0.905 (37 tiles below 0.5)
Lookup table for R12 (xBD code 0-5 -> class): [0, 1, 2, 2, 3, 255]
D5's table (xBD part 1): [0, 1, 2, 2, 3, 255] -> the same


## 3. Convert the new benchmark tiles and write the outputs
Masks keep the raw xBD codes 0–5, as D5's do, and R12 maps them with this folder's `mapping.json` when it loads them. Every tile in `manifest.json` is a benchmark image (`in_benchmark: true`). The new tiles that aren't benchmark images are only counted, for the README.

In [5]:
# ── Cell 5: Convert the new benchmark tiles (raw xBD codes 0-5) ────────────
manifest = []
for t in tqdm(new_in_bench, desc="New benchmark tiles"):
    mask, polygons = rasterize(load_label(t))
    Image.fromarray(mask).save(OUT_DIR / "masks" / f"{t}_post_disaster.png")
    pixels = np.bincount(mask.ravel(), minlength=len(XBD_CODES))
    manifest.append({
        "tile": t, "event": t.rsplit("_", 1)[0], "in_benchmark": True,
        "post_image": f"{t}_post_disaster.png", "pre_image": f"{t}_pre_disaster.png",
        "mask": f"masks/{t}_post_disaster.png",
        "polygons": polygons, "pixels": {XBD_CODES[c]: int(pixels[c]) for c in range(len(XBD_CODES))},
    })

# The tiles left out (not benchmark images): their buildings are only counted
clean_polygons = Counter()
for t in tqdm(new_clean, desc="Left-out tiles (counted only)"):
    clean_polygons.update(f["properties"].get("subtype") for f in load_label(t)["features"]["xy"])
print(f"✓ Wrote {len(manifest):,} masks to {OUT_DIR / 'masks'}")
print(f"  Left out: {len(new_clean):,} tiles holding {sum(clean_polygons.values()):,} buildings, "
      f"{clean_polygons['destroyed']:,} of them destroyed")


New benchmark tiles:   0%|          | 0/612 [00:00<?, ?it/s]

Left-out tiles (counted only):   0%|          | 0/3122 [00:00<?, ?it/s]

✓ Wrote 612 masks to D:\CSE499AB_project\Essential_task_delegation\Champion_Task\D5b-results\masks
  Left out: 3,122 tiles holding 2,455 buildings, 0 of them destroyed


In [6]:
# ── Cell 6: The pool, and the output files ─────────────────────────────────
def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: fall back to a plain-text table
        return "```\n" + df.to_string(index=False) + "\n```"

tiles = pd.DataFrame([{"tile": m["tile"], "event": m["event"], "destroyed_polygons": m["polygons"]["destroyed"]}
                      for m in manifest])
pools = pd.DataFrame([
    {"pool": "new benchmark tiles (converted for R12)", "tiles": len(tiles),
     "buildings": int(sum(sum(m["polygons"].values()) for m in manifest)),
     "destroyed_polygons": int(tiles["destroyed_polygons"].sum()),
     "tiles_with_destroyed": int((tiles["destroyed_polygons"] > 0).sum())},
    {"pool": "new tiles not in the benchmark (left out)", "tiles": len(new_clean),
     "buildings": sum(clean_polygons.values()), "destroyed_polygons": clean_polygons["destroyed"],
     "tiles_with_destroyed": "—"},
])
per_event = (tiles.groupby("event")
             .agg(tiles=("tile", "size"), destroyed_polygons=("destroyed_polygons", "sum"))
             .sort_values("tiles", ascending=False).reset_index())
print(pools.to_string(index=False))
print()
print(per_event.to_string(index=False))

with open(OUT_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=1)
with open(OUT_DIR / "mapping.json", "w") as f:
    json.dump({"lut": lut, "codes": XBD_CODES, "classes": DM3_CLASSES, "table": table.tolist(),
               "pixel_agreement": round(float(agreement), 4)}, f, indent=1)
check.to_csv(OUT_DIR / "overlap_check.csv", index=False)

(OUT_DIR / "mapping_report.md").write_text("\n".join([
    "# D5b mapping report (xBD tier3)", "",
    f"{len(check):,} tier3 tiles that DisasterM3 already uses: xBD polygons drawn by this notebook against "
    "DisasterM3's own masks. Each row shows where one xBD label's pixels go in DisasterM3.", "",
    as_markdown(mapping), "",
    f"Pixel agreement after mapping: {agreement:.1%}. "
    f"Median building IoU per tile: {check['building_iou'].median():.3f}.", "",
    f"Lookup table (xBD code 0-5 -> class, 255 = ignore): `{lut}`"
    + ("" if same_as_d5 is None else f", {'the same as' if same_as_d5 else 'DIFFERENT from'} D5's."),
]), encoding="utf-8")

config = {"task": "D5b", "smoke_test": SMOKE_TEST, "labels_dir": str(LABELS_DIR), "benchmark": str(BENCH_PATH),
          "tier3_post_labels": len(xbd_tiles), "overlap_tiles_checked": len(check), "new_tiles": len(new),
          "converted_benchmark_tiles": len(manifest), "left_out_tiles": len(new_clean),
          "bdc_questions_on_converted": bdc_hit, "lut": lut, "same_lut_as_d5": same_as_d5,
          "run_time_min": round((time.time() - t_start) / 60, 1)}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")

(OUT_DIR / "README.md").write_text("\n".join([
    "# xbd-extra-tier3 (task D5b)", "",
    f"{len(manifest):,} xBD tier3 tiles that DisasterM3's segmentation data doesn't use. All of them are DisasterM3 "
    "benchmark images (option A, chosen 19 Oct 2026), and none is one of the 645 validation images.", "",
    "## Pools", "", as_markdown(pools), "",
    f"A model trained on these tiles must not answer benchmark questions ({bdc_hit:,} of {len(bdc):,} BDC "
    "questions use them); R12 already follows that rule. The new tiles that aren't benchmark images hold "
    "no destroyed building, so they are left out.", "",
    "## Use in R12", "",
    "Upload this folder together with `D5-results/`. R12 finds each by its `mapping.json` and maps the masks the "
    "same way:", "",
    "```python",
    "lut = np.array(json.load(open(D5B / 'mapping.json'))['lut'], np.uint8)",
    "mask = lut[cv2.imread(str(D5B / m['mask']), cv2.IMREAD_GRAYSCALE)]   # 0-3, 255 = ignore",
    "```", "",
    "Images (`post_image` and `pre_image` in `manifest.json`) come from Bojan Tunguz's \"xView2 Challenge Dataset "
    "- Tier 3 Data\" on Kaggle, folder `tier3/images/`.", "",
    "## Mapping", "", f"See `mapping_report.md`. Lookup table: `{lut}`.", "",
    "## Per event", "", as_markdown(per_event),
]), encoding="utf-8")
print(f"\n✓ Wrote manifest.json, mapping.json, overlap_check.csv, mapping_report.md, config.json and README.md "
      f"({(time.time() - t_start) / 60:.1f} min)")
print("Next: upload this folder for R12 together with D5-results/.")


                                     pool  tiles  buildings  destroyed_polygons tiles_with_destroyed
  new benchmark tiles (converted for R12)    612      41820                4549                  227
new tiles not in the benchmark (left out)   3122       2455                   0                    —

             event  tiles  destroyed_polygons
    joplin-tornado    148                3274
lower-puna-volcano     60                 173
     moore-tornado     60                 208
    nepal-flooding     60                  48
   pinery-bushfire     60                  40
 portugal-wildfire     60                  61
tuscaloosa-tornado     60                 210
      woolsey-fire     60                 374
     sunda-tsunami     44                 161

✓ Wrote manifest.json, mapping.json, overlap_check.csv, mapping_report.md, config.json and README.md (2.3 min)
Next: upload this folder for R12 together with D5-results/.
